In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from itertools import product

import pandas as pd
from strat import STRAT

from dema_strat import (
  compare_benchmark_summary,
  db_connect,
  global_style,
  grid_tuning,
  load_data_and_split,
)
from exits_builder import build_be_exits
from params_finder import neighborhood_analysis
from sizer import sl_based_sizer
from visualizer import (
  combinations_compare_visualizer,
  portfolio_perf_visualizer,
  sensitivity_check_visualizer,
  trades_visualizer,
)

# CONFIG
SPLIT_RATIO = 0.7
TABLE_NAME = 'f0'
TABLE_NAME_OOS = f"{TABLE_NAME}_OOS"
DATA_PATH = '~/market_data/XAUUSD/XAUUSD_M15.csv'
TARGET = 'profit_factor'
RISK_PER_TRADE = 10

PARAMS = {
    "ema_period": {"type": "INTEGER", "data": range(5, 15), "default": 9},
    "smoothing_period": {"type": "INTEGER", "data": range(6, 16), "default": 12},
    "use_vwap": {"type": "BOOL", "data": [True, False], "default": True},
    "atr_period": {"type": "INTEGER", "data": range(14, 22, 2), "default": 18},
    "sl_mult": {"type": "REAL", "data": range(1,3), "default": 1},
    "rr_ratio": {"type": "REAL", "data": [0.25, 0.5, 1.0, 1.5, 2.0], "default": 1},
}

PARAM_COLS = list(PARAMS)
PARAMS_GRID = list(product(*(v["data"] for v in PARAMS.values())))
PARAMS_DEFAULT = {
    k: v.get("default", v["data"][0]) 
    for k, v in PARAMS.items()
}

TOP_N = max(100, len(PARAMS_GRID))

conn, cursor = db_connect()

def run_backtest(data, params = PARAMS_DEFAULT):
  # build the strat
  strat = STRAT(data, params)

  rr_ratio = params.get("rr_ratio", 2)
  sl_mult = params.get("sl_mult", 1)

  # exits and sizer
  long_exits, short_exits, sl_dist = build_be_exits(data, strat.signals, rr_ratio=rr_ratio, sl_mult=sl_mult)
  sizer = sl_based_sizer(risk_per_trade=RISK_PER_TRADE, sl_dists=sl_dist)

  portfolio = strat._run_backtest(long_exits, short_exits, size=sizer, size_type="amount")
  return portfolio, strat

df_ios, df_oos = load_data_and_split(DATA_PATH, split_ratio=SPLIT_RATIO)

## Grid tuning

In [ ]:
grid_tuning(params=PARAMS, table_name=TABLE_NAME, data=df_ios, run_backtest=run_backtest, replace_table=True)
grid_tuning(params=PARAMS, table_name=f"{TABLE_NAME}_OOS", data=df_oos, run_backtest=run_backtest, replace_table=True)

## Combinations compare for TOP 200 vs All combinations

In [ ]:
def get_query(limit = True):
    return f"""
        SELECT *
        FROM {TABLE_NAME}
        ORDER BY {TARGET} DESC
        {f'LIMIT {TOP_N}' if limit else ''}
    """

top_df = pd.read_sql_query(get_query(), conn)
all_df = pd.read_sql_query(get_query(limit = False), conn)

print(f"Top {TOP_N} combinations")
display(top_df)

combinations_compare_visualizer(
  first_data=top_df,
  second_data=all_df,
  chart_title=f"TOP {TOP_N} vs All combinations",
  first_name="TOP 200",
  second_name="All combinations"
)

## Combinations compare for OOS vs IOS

In [ ]:

def get_query(table):
    return f"""
        SELECT *
        FROM {table}
        ORDER BY {TARGET} DESC
        LIMIT {TOP_N}
    """

ios = pd.read_sql_query(get_query(TABLE_NAME), conn)
oos = pd.read_sql_query(get_query(f"{TABLE_NAME}_OOS"), conn)

print(f"OOS combinations (TOP {TOP_N})")
display(oos)

combinations_compare_visualizer(
  first_data=oos,
  second_data=ios,
  chart_title=f"OOS vs IOS (TOP {TOP_N})",
  first_name="OOS",
  second_name="IOS"
)

## Parameter neighborhood analysis 

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

best_combination = neighborhood_analysis(df, PARAMS, TARGET=TARGET)

## Benchmark

In [ ]:
best_combination_param = best_combination[PARAM_COLS]
print(best_combination_param)

portfolio_ios, strat_ios = run_backtest(df_ios, params=best_combination_param)
portfolio_oos, _ = run_backtest(df_oos, params=best_combination_param)

compare_benchmark_summary(
  data_ios=df_ios,
  data_oos=df_oos,
  portfolio_ios=portfolio_ios,
  portfolio_oos=portfolio_oos,
)

## Trade and Portfolio performance

In [ ]:
trades_visualizer(data=df_ios, signals=strat_ios.signals, portfolio=portfolio_ios, N = 500)
portfolio_perf_visualizer(portfolio_ios)

## Sensitivity check

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

sensitivity_check_visualizer(df, PARAM_COLS=PARAM_COLS, TARGET=TARGET)

## Calc the EV for propfirm trading (TOPSTEPX and FTMO)